# 📓 Lab 00 — Machine Learning clásico: supervisado, no supervisado y reforzado

**Sesión:** [Sesión 0 — Fundamentos de Machine Learning](../sesiones/00-fundamentos-ml.md)

Tres experiencias en un solo laboratorio, una por paradigma:

1. **Supervisado:** cinco modelos clásicos compitiendo sobre `make_moons` — el mismo dataset que una red neuronal atacará en el Lab 02.
2. **No supervisado:** k-means agrupando SIN etiquetas (y cayendo en una trampa instructiva) + PCA comprimiendo 64 píxeles a 2 números.
3. **Reforzado:** un agente Q-learning que aprende a llegar a una meta esquivando un pozo, en numpy puro.

**Evidencia a entregar:** tabla comparativa de los 5 modelos, fronteras de decisión, la política aprendida por tu agente y una conclusión de ≤100 palabras.

In [ ]:
# Setup: solo numpy, matplotlib y scikit-learn (ya están en requirements.txt)
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)          # reproducibilidad: mismos números en cada corrida
print('numpy:', np.__version__)

## 1. El dataset y el baseline obligatorio

`make_moons` genera dos "lunas" entrelazadas: imposibles de separar con una línea recta — perfecto para ver la *personalidad* de cada modelo.

**Regla de oro del curso:** antes de cualquier modelo, un **baseline**. El más barato es *majority class*: predecir siempre la clase más común. Cualquier modelo que no lo supere con claridad no está aportando nada.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

X, y = make_moons(n_samples=500, noise=0.22, random_state=42)

# Split estratificado: misma proporción de clases en train y test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

# Baseline majority: ¿qué accuracy da predecir SIEMPRE la clase más común?
clase_mayoritaria = np.bincount(y_train).argmax()
acc_majority = (y_test == clase_mayoritaria).mean()
print(f"Baseline majority: predice siempre '{clase_mayoritaria}' "
      f"→ accuracy = {acc_majority:.3f}")

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='RdBu_r',
           edgecolors='k', s=25)
ax.set_title('make_moons: el dataset que nos acompañará hasta el Lab 02')
plt.show()

## 2. Cinco modelos supervisados, un mismo contrato

Todos los modelos de scikit-learn comparten la misma API — `fit(X, y)` para entrenar, `predict(X)` para responder — así que compararlos es un bucle.

> 🧩 **Predice antes de ejecutar:** ¿cuál crees que gana en moons? ¿la regresión logística (lineal), k-NN (vecinos), el árbol (preguntas si/no), el bosque o la SVM? Escribe tu apuesta.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

modelos = {
    'Regresión logística': LogisticRegression(),                # lineal (la neurona de la Sesión 1)
    'k-NN (k=15)': KNeighborsClassifier(n_neighbors=15),        # "dime tus vecinos"
    'Árbol (prof. 5)': DecisionTreeClassifier(max_depth=5, random_state=0),
    'Random forest': RandomForestClassifier(n_estimators=100, random_state=0),
    'SVM (RBF)': SVC(gamma=2, probability=True, random_state=0),  # margen máximo + kernel
}

resultados = {}
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)              # entrenar SOLO con train
    pred = modelo.predict(X_test)             # evaluar SOLO en test
    resultados[nombre] = {
        'accuracy': (pred == y_test).mean(),
        'f1': f1_score(y_test, pred),
    }

print(f"{'modelo':<22} {'accuracy':>9} {'F1':>7}")
print('-' * 40)
print(f"{'Majority (baseline)':<22} {acc_majority:>9.3f} {'—':>7}")
for nombre, m in resultados.items():
    print(f"{nombre:<22} {m['accuracy']:>9.3f} {m['f1']:>7.3f}")

### Las fronteras de decisión: la "personalidad" de cada familia

La tabla dice *cuánto* acierta cada modelo; la frontera dice *cómo piensa*. Observa: la logística traza UNA recta; el árbol y el bosque hacen escaleras (cortes paralelos a los ejes); k-NN y la SVM dibujan curvas suaves.

In [ ]:
# Malla de puntos para pintar la probabilidad de clase 1 en todo el plano
xx, yy = np.meshgrid(
    np.linspace(X[:, 0].min() - 0.6, X[:, 0].max() + 0.6, 200),
    np.linspace(X[:, 1].min() - 0.6, X[:, 1].max() + 0.6, 200),
)
malla = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(2, 3, figsize=(13, 7.5))
axes[0, 0].scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='RdBu_r',
                   edgecolors='k', s=18)
axes[0, 0].set_title('los datos (test)')

for ax, (nombre, modelo) in zip(axes.ravel()[1:], modelos.items()):
    prob = modelo.predict_proba(malla)[:, 1].reshape(xx.shape)
    ax.contourf(xx, yy, prob, levels=16, cmap='RdBu_r', alpha=0.5)
    ax.contour(xx, yy, prob, levels=[0.5], colors='k', linewidths=1.5)
    ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='RdBu_r',
               s=12, edgecolors='k', linewidths=0.3)
    ax.set_title(f"{nombre} — acc {resultados[nombre]['accuracy']:.2f}")

for ax in axes.ravel():
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

## 3. No supervisado: k-means sin etiquetas

Ahora **escondemos las etiquetas**: k-means solo ve los puntos. Lo probamos en dos mundos:

- sobre **blobs** (grupos redondos) — su terreno natural;
- sobre **moons** — donde su suposición de "grupos redondos alrededor de un centroide" lo traiciona.

> La lección: todo modelo lleva **suposiciones incorporadas**; conocerlas es saber cuándo usarlo. (En la Sesión 2 esta idea vuelve con nombre propio: *inductive bias*.)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

X_blobs, _ = make_blobs(n_samples=400, centers=3, cluster_std=0.9,
                        random_state=4)

# n_init=10: k-means arranca de centroides al azar; se queda con el mejor de 10 intentos
km_blobs = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X_blobs)
km_moons = KMeans(n_clusters=2, n_init=10, random_state=0).fit(X)   # ¡sin y!

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.scatter(X_blobs[:, 0], X_blobs[:, 1], c=km_blobs.labels_,
            cmap='viridis', s=15)
ax1.scatter(*km_blobs.cluster_centers_.T, marker='X', s=250, c='red',
            edgecolors='k')
ax1.set_title('k-means sobre blobs: su terreno natural ✓')
ax2.scatter(X[:, 0], X[:, 1], c=km_moons.labels_, cmap='viridis', s=15)
ax2.scatter(*km_moons.cluster_centers_.T, marker='X', s=250, c='red',
            edgecolors='k')
ax2.set_title('k-means sobre moons: parte las lunas por la mitad ✗')
plt.show()

### PCA: comprimir 64 píxeles a 2 números

Cada dígito manuscrito de `load_digits` es una imagen 8×8 = **64 features**. PCA encuentra las 2 direcciones donde los datos más varían y proyecta todo ahí. El color es el dígito real — PCA **nunca lo vio**: si los colores quedan agrupados, la compresión preservó estructura.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA

digits = load_digits()
print('shape original  :', digits.data.shape)      # (1797, 64)

pca = PCA(n_components=2)
Z = pca.fit_transform(digits.data)
print('shape comprimida:', Z.shape)                # (1797, 2)
print('varianza explicada por las 2 componentes:',
      pca.explained_variance_ratio_.sum().round(3))

fig, ax = plt.subplots(figsize=(7, 5.5))
sc = ax.scatter(Z[:, 0], Z[:, 1], c=digits.target, cmap='tab10', s=10)
ax.set_xlabel('componente principal 1')
ax.set_ylabel('componente principal 2')
ax.set_title('1797 dígitos: de 64 píxeles a 2 números por dígito')
plt.colorbar(sc, label='dígito real (PCA nunca lo vio)')
plt.show()

## 4. Reforzado: un agente que aprende a llegar a la meta

Un **gridworld 4×4**: el agente nace en la esquina superior izquierda, la META (+1) está en la inferior derecha y hay un POZO (−1) en el camino. Cada paso cuesta −0.02 (para que no vagabundee).

El agente aprende con **Q-learning**: una tabla `Q[fila, col, acción]` que estima la recompensa futura de cada acción en cada casilla, actualizada tras cada paso con

$$Q(s,a) \leftarrow Q(s,a) + \alpha\left[r + \gamma \max_{a'} Q(s',a') - Q(s,a)\right]$$

*Lectura: "ajusta tu estimación hacia lo que viviste — la recompensa inmediata r más lo mejor que promete la casilla donde caíste (descontado por γ)". α es el learning rate: el mismo concepto que reencontrarás en la Sesión 1.*

Y el dilema **explorar vs explotar** se resuelve con ε-greedy: al inicio actúa casi al azar (explora); con la experiencia, ε baja y explota lo aprendido.

In [ ]:
# ══ El mundo ══════════════════════════════════════════════════════
N = 4                              # tablero de N×N casillas
META, POZO = (3, 3), (1, 2)        # (fila, columna)
ACCIONES = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # ↑ ↓ ← →
FLECHAS = ['↑', '↓', '←', '→']

# ══ Hiperparámetros del agente ════════════════════════════════════
ALPHA = 0.2       # learning rate: cuánto corrijo la tabla en cada paso
GAMMA = 0.95      # descuento: cuánto valoro el futuro frente al presente
EPISODIOS = 400   # cuántas "vidas" juega el agente

rng = np.random.default_rng(0)
Q = np.zeros((N, N, 4))            # la tabla Q: una fila de 4 valores por casilla
recompensas = []                   # para graficar el aprendizaje

for episodio in range(EPISODIOS):
    fila, col = 0, 0               # cada episodio nace en el inicio
    total = 0.0

    for paso in range(60):         # límite de pasos por episodio
        # ── ε-greedy: explorar al inicio, explotar con la experiencia ──
        epsilon = max(0.05, 1 - episodio / 250)
        if rng.random() < epsilon:
            accion = rng.integers(4)                 # explorar: acción al azar
        else:
            accion = int(np.argmax(Q[fila, col]))    # explotar: la mejor conocida

        # ── El entorno responde: nueva casilla y recompensa ──
        df, dc = ACCIONES[accion]
        fila2 = min(max(fila + df, 0), N - 1)        # chocar con la pared = quedarse
        col2 = min(max(col + dc, 0), N - 1)
        if (fila2, col2) == META:
            recompensa, termina = 1.0, True
        elif (fila2, col2) == POZO:
            recompensa, termina = -1.0, True
        else:
            recompensa, termina = -0.02, False       # cada paso cuesta un poquito

        # ── La actualización de Q-learning (la fórmula de arriba) ──
        objetivo = recompensa + GAMMA * np.max(Q[fila2, col2])
        Q[fila, col, accion] += ALPHA * (objetivo - Q[fila, col, accion])

        total += recompensa
        fila, col = fila2, col2
        if termina:
            break

    recompensas.append(total)

# ── La curva de aprendizaje: la recompensa sube con la experiencia ──
ventana = 20
suave = np.convolve(recompensas, np.ones(ventana) / ventana, mode='valid')
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(recompensas, alpha=0.3, label='recompensa por episodio')
ax.plot(range(ventana - 1, EPISODIOS), suave, linewidth=2.5,
        label=f'promedio móvil ({ventana})')
ax.set_xlabel('episodio')
ax.set_ylabel('recompensa total')
ax.set_title('El agente aprende probando')
ax.legend()
plt.show()

In [ ]:
# ── La política aprendida: la mejor acción en cada casilla ──
print('Política aprendida:\n')
for fila in range(N):
    linea = []
    for col in range(N):
        if (fila, col) == META:
            linea.append(' META')
        elif (fila, col) == POZO:
            linea.append(' POZO')
        else:
            linea.append(f'  {FLECHAS[int(np.argmax(Q[fila, col]))]}  ')
    print(''.join(linea))

# ── Seguir la política desde el inicio (sin explorar) ──
fila, col, camino = 0, 0, [(0, 0)]
for _ in range(20):
    if (fila, col) in (META, POZO):
        break
    df, dc = ACCIONES[int(np.argmax(Q[fila, col]))]
    fila = min(max(fila + df, 0), N - 1)
    col = min(max(col + dc, 0), N - 1)
    camino.append((fila, col))

print('\nCamino greedy desde el inicio:', ' → '.join(map(str, camino)))
assert camino[-1] == META, 'el agente debería llegar a la meta'
print('✔ el agente llega a la META esquivando el pozo')

## 5. 🎯 Reto y entrega

**Experimentos (elige dos, cambia UNA variable por experimento):**

1. *Supervisado:* varía `k` en k-NN (1 vs 50) o `max_depth` del árbol (2 vs 20). ¿Qué le pasa a la frontera y al accuracy de test? (Estás viendo under/overfitting antes de que la Sesión 1 le ponga nombre.)
2. *No supervisado:* pide `n_clusters=4` sobre los blobs de 3 grupos. ¿Qué inventa k-means?
3. *Reforzado:* mueve el `POZO` a (2, 1), o baja `GAMMA` a 0.5. ¿Cómo cambia la política? ¿Por qué un agente "cortoplacista" (γ bajo) toma peores rutas?

**Evidencia a entregar:**

- Tabla comparativa de los 5 modelos + baseline.
- Fronteras de decisión y tu apuesta inicial: ¿acertaste?
- La política de tu agente (con tu experimento aplicado).
- Conclusión ≤100 palabras: ¿qué modelo elegirías para moons y por qué?
- Commit: `feat: complete ml fundamentals lab`.

**Siguiente parada:** [Sesión 1](../sesiones/01-fundamentos.md) — una red neuronal ataca este mismo dataset, y vas a poder comparar contra los números de HOY.